pre ove sveske **OBAVEZNO** pokrenuti features.ipynb 

In [192]:
import pandas as pd
from collections import defaultdict

In [193]:
import warnings
warnings.filterwarnings("ignore")

In [194]:
#ucitavanje neophodnih tabela
pits = pd.read_csv('tables//processed_pitstop_data.csv')
all_data = pd.read_csv('tables//trke_sa_driverId.csv')

In [195]:
all_data = all_data.drop('_merge', axis=1)

In [196]:
all_data.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', 'driverId', 'race_name', 'circuit', ' lap_length',
       ' number_of_laps', ' number_of_corners', 'AirTemp', 'Humidity',
       'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin', 'wind_direction',
       'is_sprint_weekend', 'last_year_time', 'avg_position_last5',
       'num_dnfs_last5', 'avg_gained_lost_last5'],
      dtype='object')

In [197]:
pits.columns

Index(['season', 'round', 'lap', 'driverId', 'stop', 'duration', 'time'], dtype='object')

In [198]:
pits = pits.drop('time', axis=1)

In [199]:
#izostavljanje nepotrebnih sezona
pits = pits[pits['season'] != 2015]
pits = pits[pits['season'] != 2016]
pits = pits[pits['season'] != 2017]

In [200]:
#koje su sezone u tabelama
seasons = list(set(zip(pits['season'])))

In [201]:
#posto prethodna linija daje listu vrednosti oblika [(1,),(2.)], ovaj kod pretvara u listu [1, 2]
for i in range(len(seasons)):
    seasons[i] = (seasons[i][0])
seasons = sorted(seasons)

In [202]:
print(seasons)

[2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [203]:
# koliko rundi ima po sezoni
# ako nema zip onda ne mora ovaj for
num_rounds = []
for season in seasons:
    #posto prethodna linija daje listu vrednosti oblika [(1,),(2.)], ovaj kod pretvara u listu [1, 2]
    one_season_rounds = pits[pits['season'] == season]
    rounds = list(set(zip(one_season_rounds['round'])))
    for i in range(len(rounds)):
        rounds[i] = (rounds[i][0])
    num_rounds.append(len(rounds))
    # print(num_rounds)
    

    

In [204]:
pits.columns

Index(['season', 'round', 'lap', 'driverId', 'stop', 'duration'], dtype='object')

In [ ]:
df = pd.DataFrame(columns=['season', 'round', 'driverId', 'num_pit_stops', 'lap_pit_stops', 'duration_pit_stops'])
for i in range(len(seasons)):
    for j in range(num_rounds[i]):
        one_race = pits[(pits['season'] == seasons[i]) & (pits['round'] == j+1)]
        # ovde se izracunavaju broj stajanja, duzina svakog i u kom krugu se odigralO
        season = seasons[i]
        round = j+1

        one_race = one_race.drop(['season', 'round'], axis=1)
        # pravi dictonary primer: 'alonso': [{'lap': 26, 'stop': 1, 'duration': 22.573}] 
        
        pits["duration"] = pd.to_numeric(pits["duration"], errors="coerce")


        pit_stops_dict = one_race.groupby('driverId').apply(lambda g: g.drop(columns='driverId').to_dict(orient='records')).to_dict()
        
       # izvlaci u kom se krugu odigrao pit i koliko je trajao
        for driver in pit_stops_dict:
            one_driver = pit_stops_dict[driver]
            num = len(one_driver)
            lap, duration = [], []
            for x in one_driver:
                lap.append(x['lap'])
                duration.append(x['duration'])
            df.loc[len(df)] = [season, round, driver, num, lap, duration]

In [223]:
df

,season,round,driverId,num_pit_stops,lap_pit_stops,duration_pit_stops
0,2018,1,alonso,1,[26],[22.573]
1,2018,1,bottas,1,[25],[21.664]
2,2018,1,brendon_hartley,2,"[1, 22]","[22.213, 22.296]"
3,2018,1,grosjean,1,[24],[23.054]
4,2018,1,hamilton,1,[19],[21.821]
...,...,...,...,...,...,...
2766,2024,24,russell,1,[26],[21.656]
2767,2024,24,sainz,1,[25],[21.28]
2768,2024,24,stroll,2,"[11, 32]","[23.266, 22.053]"
2769,2024,24,tsunoda,1,[22],[21.869]


In [207]:
#ucitavanje podataka o svim krugovima
laps = pd.read_csv('tables\processed_lap_data.csv')

In [208]:
laps.columns

Index(['season', 'round', 'lap', 'position', 'driverId'], dtype='object')

In [209]:
pits_data = df

In [210]:
pits_data.columns

Index(['season', 'round', 'driverId', 'num_pit_stops', 'lap_pit_stops',
       'duration_pit_stops'],
      dtype='object')

In [211]:
pits_data = pits_data.drop(['duration_pit_stops', 'num_pit_stops'], axis=1)

In [ ]:
# pravi dictoray svih pitoa koji su se odigrali podeljeni po rundi i krugu
pit_stops_dict = defaultdict(set)

for _, row in pits_data.iterrows():
    if type(row['lap_pit_stops'])==list:
        for i in range(len(row['lap_pit_stops'])):
            key = (row['season'], row['round'], row['lap_pit_stops'][i]) 
    pit_stops_dict[key].add(row['driverId'])

pits_dict = dict(pit_stops_dict)

In [ ]:
# pronalazak DNFova

# poseldnji krug trke
race_last_laps = laps.groupby(['season', 'round'])['lap'].max().reset_index()
race_last_laps = race_last_laps.rename(columns={'lap': 'max_lap'})

# poslednji krug svakog vozaca
driver_last_laps = laps.groupby(['season', 'round', 'driverId'])['lap'].max().reset_index()


merged = pd.merge(driver_last_laps, race_last_laps, on=['season', 'round'])

# Filtrirati vozace koji su presli manje od ukupnog broja krugova
dnfs = merged[merged['lap'] < merged['max_lap']]

# napraviti dictionary
dnfs_dict = defaultdict(set)

for _, row in dnfs.iterrows():
    # uzima se status svakog vozaca
    value = all_data.loc[(all_data['season'] == row['season']) & 
                         (all_data['round'] == row['round']) & 
                         (all_data['driverId'] == row['driverId']), 'status'].iloc[0]
    # ako status ne sadrzi finished ili lap onda vozac nije zavrsio trku
    if not("finnished".lower() in value.lower() or "lap".lower() in value.lower()):
        key = (row['season'], row['round'], row['lap'])  # krug kada su se dnf-ovali
        dnfs_dict[key].add(row['driverId'])

dnfs_dict = dict(dnfs_dict)


In [214]:
print("len(seasons):", len(seasons))
print("seasons:", seasons)


len(seasons): 7
seasons: [2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [ ]:
#RACUNANJE BROJA PRETICANJA

overtakes_list = []

for i in range(len(seasons)):
    for j in range(num_rounds[i]):
        season = seasons[i]
        round_num = j + 1

        race_laps = laps[(laps['season'] == season) & (laps['round'] == round_num)]
        drivers = race_laps['driverId'].unique()

        pivoted = race_laps.pivot(index='lap', columns='driverId', values='position').sort_index()
        max_lap = pivoted.index.max()

        #racunanje razlike pozicija izmedju prethodnog i trenutnog kruga
        pivot_diff = pivoted.diff()
        potential_overtakes = pivot_diff.stack()
        potential_overtakes = potential_overtakes[potential_overtakes < 0].reset_index()
        potential_overtakes.columns = ['lap', 'driverId', 'position_change']
        
        
        driver_overtakes = defaultdict(int)

        for _, row in potential_overtakes.iterrows():
            lap = row['lap']
            driver_overtaker = row['driverId']
            delta = row['position_change']
            position = pivoted.loc[lap, driver_overtaker]
            delta = abs(delta)
            

            # Provera da li je vozac ispred otisao na promenu guma ili se dnf-ovao
            for driver in drivers:
                if delta <= 0 or lap >= max_lap-1:
                    break  # nema vise potencijalnih preticanja ili je poslednji krug
                    
                if pivoted.loc[lap, driver] < position:
                    try:
                        # smanjiti delta ako se vozac ispred dnf-ovao
                        if driver in dnfs_dict.get((season, round_num, lap), set()):
                            delta -= 1
                        # smanjiti delta ako se vozac ispred pit-ovao
                        elif (driver in pits_data.get((season, round_num, lap), set())) and (pivoted.loc[lap+1, driver] > pivoted.loc[lap + 1, driver_overtaker]):
                            delta -= 1
                    except Exception as e:
                        print(f"Error at season {season}, round {round_num}, lap {lap}: {e}")

                    

            if delta > 0:
                driver_overtakes[driver_overtaker] += delta

        # dodavanje za trenutnu trku
        for driver, num in driver_overtakes.items():
            overtakes_list.append({'season': season, 'round': round_num, 'driverId': driver, 'num_overtakes': num})

overtakes = pd.DataFrame(overtakes_list)


In [217]:
overtakes['num_overtakes'] = overtakes['num_overtakes'].fillna(0)


In [218]:
overtakes.head()

,season,round,driverId,num_overtakes
0,2018,1,bottas,6.0
1,2018,1,ricciardo,4.0
2,2018,1,brendon_hartley,5.0
3,2018,1,gasly,1.0
4,2018,1,leclerc,5.0


In [219]:
all_data = pd.merge(all_data, overtakes, on=['season', 'round', 'driverId'], how='outer')

In [220]:
len(all_data)

2979

In [222]:
all_data.to_csv('tables/trkeDID_pits_over.csv', index=False)